In [1]:
from spacy.lang.en import English 

nlp = English()

#Add a sentencizer pipeline
nlp.add_pipe("sentencizer")
# sentencizer breaks a group of sentences into individual sentences

# Create a document instance as an exemple
doc = nlp("This is a sentence. This is another sentence. This is a third sentence.")
print(type(doc))
assert len(list(doc.sents)) == 3

# Acces the sentences of the document
list(doc.sents)

<class 'spacy.tokens.doc.Doc'>


[This is a sentence., This is another sentence., This is a third sentence.]

In [2]:
import pickle
import os

data_dir = "../data"
file_path = os.path.join(data_dir, "pages_and_texts.pkl")

#load the pages_and_texts list
try:    
    with open(file_path, "rb") as f:
        pages_and_texts = pickle.load(f)
except Exception as e:
    print(f"Error loading file: {e}")

print(f"First page: {pages_and_texts[0]}")

First page: {'page_number': -41, 'page_char_count': 29, 'page_word_count': 4, 'page_sentence_count_raw': 1, 'page_token_count': 7.25, 'text': 'Human Nutrition: 2020 Edition'}


In [3]:
from tqdm.auto import tqdm

for page_dict in tqdm(pages_and_texts):
    # for every page, create a list with its the sentences
    # create a new key "sentences" and assign it the list of sentences
    page_dict["sentences"] = list(nlp(page_dict["text"]).sents)

    # make sure all sentences are strings
    page_dict["sentences"] = [str(sent) for sent in page_dict["sentences"]]

    page_dict["page_sentences_count_spacy"] = len(page_dict["sentences"])


  0%|          | 0/1208 [00:00<?, ?it/s]

In [4]:
import pandas as pd

df = pd.DataFrame(pages_and_texts)
print(df.describe().round(2))


       page_number  page_char_count  page_word_count  page_sentence_count_raw  \
count      1208.00          1208.00          1208.00                  1208.00   
mean        562.50          1148.00           198.30                     9.97   
std         348.86           560.38            95.76                     6.19   
min         -41.00             0.00             1.00                     1.00   
25%         260.75           762.00           134.00                     4.00   
50%         562.50          1231.50           214.50                    10.00   
75%         864.25          1603.50           271.00                    14.00   
max        1166.00          2308.00           429.00                    32.00   

       page_token_count  page_sentences_count_spacy  
count           1208.00                     1208.00  
mean             287.00                       10.32  
std              140.10                        6.30  
min                0.00                        0.00  


On average each of our pages has 10 sentences and an total average of 287 token per page.

So a group with 10 sentences will give plenty of room for the text to embedded by 'all-mpnet-base-v2' model which has a capacity of 384 tokens.

To split our sentences into chunks of 10 or less, we create a function which recursively breaks down into sublists of a specified size.

In [5]:
#split size
num_sentences_chunk_size = 10

#recursive function to split a list into desired sizes
def split_list(list, slice_size = int) -> list[list[str]]:
    """
    Recursively split a list into sublists of a specified size.

    For example, a list of 17 sentences will be split into 2 sublists of [10] and [7]
    """
    final_list = []
    for i in range(0, len(list), slice_size):
        final_list.append(list[i:i+slice_size])
    return final_list

for page in pages_and_texts:
    page["sentences_chunks"] = split_list(page["sentences"], num_sentences_chunk_size)
    page["num_chunks"] = len(page["sentences_chunks"])


### Splitting each chunk into its own item
Before: each chunk is within his page

After: each chunk is an own item referring to his page

Create a new list of dictionaires each containing a single chunk of sentences with relative information

In [39]:
import re

pages_and_chunks = []
for page in pages_and_texts:
    for sentence_chunk in page["sentences_chunks"]:
        chunk_dict = {}
        chunk_dict["page_number"] = page["page_number"]

        #Join the sentence together into a paragraph-like structure, aka a chunk(so they are a single strin)
        joined_sentence_chunk = "".join(sentence_chunk).replace("  "," ").strip() #every sentence in one string(no spaces)
        # ".A" -> ". A" for any full-stop/capital letter combo
        joined_sentence_chunk = re.sub(r'\.([A-Z])', r'. \1', joined_sentence_chunk)
        chunk_dict["sentence_chunk"] = joined_sentence_chunk

        #Get stats about the chunk
        chunk_dict["chunk_char_count"] = len(joined_sentence_chunk)
        chunk_dict["chunk_word_count"] = len([word for word in joined_sentence_chunk.split(" ")])
        chunk_dict["chunk_token_count"] = len(joined_sentence_chunk) / 4
        chunk_dict["chunk_sentence_count"] = len(list(nlp(joined_sentence_chunk).sents))

        pages_and_chunks.append(chunk_dict)
    
#How many chunks do we have?
print(len(pages_and_chunks))


1843


In [64]:
import random
#view a random sample
random.sample(pages_and_chunks, k=1)


[{'page_number': 465,
  'sentence_chunk': '“Aerobic Production Pathways” by Boumphreyfr / CC BY-SA 3.0 • Stage 1. Glycolysis for glucose, β-oxidation for fatty acids, or amino-acid catabolism • Stage 2. Citric Acid Cycle (or Krebs cycle) • Stage 3. Electron Transport Chain and ATP synthesis Figure 8.4 ATP Production Pathway The breakdown of glucose begins with glycolysis, which is a ten- step metabolic pathway yielding two ATP per glucose molecule; glycolysis takes place in the cytosol and does not require oxygen. In addition to ATP, the end-products of glycolysis include two three- carbon molecules, called pyruvate. Pyruvate can either be shuttled to the citric acid cycle to make more ATP or follow an anabolic pathway. If a cell is in negative-energy balance, pyruvate is transported to the mitochondria where it first gets one of its carbons chopped off, yielding acetyl-CoA. The breakdown of fatty acids begins with the catabolic pathway, known as β-oxidation, which takes place in the m

##### Now we're broken our whole textbook into chunks of 10 sentences or less as well as the page number they came from

##### WE check that the size of each chunk is less than the input embedding dimension required by our embeddings model(mpnet 384)

In [65]:
df = pd.DataFrame(pages_and_chunks)
print(df.shape)
print(df.describe().round(2))
print(df.info())

(1843, 6)
       page_number  chunk_char_count  chunk_word_count  chunk_token_count  \
count      1843.00           1843.00           1843.00            1843.00   
mean        583.38            734.44            112.33             183.61   
std         347.79            447.54             71.22             111.89   
min         -41.00             12.00              3.00               3.00   
25%         280.50            315.00             44.00              78.75   
50%         586.00            746.00            114.00             186.50   
75%         890.00           1118.50            173.00             279.62   
max        1166.00           1831.00            297.00             457.75   

       chunk_sentence_count  
count               1843.00  
mean                   6.58  
std                    3.23  
min                    1.00  
25%                    4.00  
50%                    7.00  
75%                   10.00  
max                   11.00  
<class 'pandas.core.frame.

In [66]:
print(df.columns)

Index(['page_number', 'sentence_chunk', 'chunk_char_count', 'chunk_word_count',
       'chunk_token_count', 'chunk_sentence_count'],
      dtype='object')


In [69]:
min_token_lenth = 30
for row in df[df["chunk_token_count"] <= min_token_lenth].sample(5).iterrows():
    print(f'Chunk token count: {row[1]["chunk_token_count"]} | Text: {row[1]["sentence_chunk"]} | Sentence count: {row[1]["chunk_sentence_count"]}')

Chunk token count: 12.5 | Text: https://www.fda.gov/food/ 1022 | Food Preservation | Sentence count: 1
Chunk token count: 25.25 | Text: The Polynesian Family System in Ka-‘u. Rutland, Vermont: Charles E. Tuttle Company 780 | Introduction | Sentence count: 2
Chunk token count: 24.5 | Text: view it online here: http://pressbooks.oer.hawaii.edu/ humannutrition2/?p=354 Phytochemicals | 605 | Sentence count: 1
Chunk token count: 21.5 | Text: http://www.health.gov.fj/?page_id=1406. Accessed November 12, 2017. 652 | Introduction | Sentence count: 3
Chunk token count: 29.75 | Text: 2011.  https://www.ers.usda.gov/publications/pub- details/?pubid=44909. Accessed April 15, 2018. 1138 | Food Insecurity | Sentence count: 4


##### Looks like many of these are headers and footers of different pages, which dont offer to much information.

##### We filter to include only the chunks with over 30 tokens in length

In [72]:
pages_and_chunks_over_min_token_len = df[df["chunk_token_count"] > min_token_lenth].to_dict(orient="records")
print(type(pages_and_chunks_over_min_token_len))
print(pages_and_chunks_over_min_token_len[:2])
print(f"Number of chunks: {len(pages_and_chunks_over_min_token_len)}")

<class 'list'>
[{'page_number': -39, 'sentence_chunk': 'Human Nutrition: 2020 Edition UNIVERSITY OF HAWAI‘I AT MĀNOA FOOD SCIENCE AND HUMAN NUTRITION PROGRAM ALAN TITCHENAL, SKYLAR HARA, NOEMI ARCEO CAACBAY, WILLIAM MEINKE-LAU, YA-YUN YANG, MARIE KAINOA FIALKOWSKI REVILLA, JENNIFER DRAPER, GEMADY LANGFELDER, CHERYL GIBBY, CHYNA NICOLE CHUN, AND ALLISON CALABRESE', 'chunk_char_count': 308, 'chunk_word_count': 42, 'chunk_token_count': 77.0, 'chunk_sentence_count': 1}, {'page_number': -38, 'sentence_chunk': 'Human Nutrition: 2020 Edition by University of Hawai‘i at Mānoa Food Science and Human Nutrition Program is licensed under a Creative Commons Attribution 4.0 International License, except where otherwise noted.', 'chunk_char_count': 210, 'chunk_word_count': 30, 'chunk_token_count': 52.5, 'chunk_sentence_count': 1}]
Number of chunks: 1680


In [165]:
%%time

#'all-mpnet-base-v2 take as input maxim 384 words and outputs a veector with 768 dimensions
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2", device="mps")

sentences = [
    "The sentence transformers library provides an easy an open-source way to create embeddings.",
    "Sentences can be embedded one by one or a list of strins",
    "Embeddings are one of the mst powerful concepts in machine learning!",
    "Learn to use embeddins well and you'll be well on your way to being an AI engineer."
]

#Sentece are encoded/embedded by calling model.encode()
embeddings = embedding_model.encode(sentences)
embeddings_dict = dict(zip(sentences, embeddings))


CPU times: user 144 ms, sys: 389 ms, total: 533 ms
Wall time: 4.69 s


In [191]:
print(f"Shape of the embeddings: {embeddings.shape}")
for sentence, embedding in embeddings_dict.items():
    print("Sentence:", sentence)
    print("Embedding:", embedding)
    print("")

Shape of the embeddings: torch.Size([1680, 768])
Sentence: The sentence transformers library provides an easy an open-source way to create embeddings.
Embedding: [-2.23365184e-02  4.42344137e-02 -1.50433052e-02  6.52768388e-02
 -2.30782218e-02 -1.38487834e-02 -4.05537011e-03 -5.58167659e-02
  1.21006314e-02 -2.94471141e-02  3.13122086e-02  4.23788913e-02
 -5.71415052e-02  2.94889230e-02  3.29702571e-02 -4.69152704e-02
  4.38281298e-02 -5.00975584e-04 -1.39744328e-02  1.27507504e-02
  4.76005971e-02  4.24170941e-02  1.97268501e-02  5.10939583e-02
 -1.61202885e-02 -3.39792408e-02  4.09669382e-03 -2.67526843e-02
  4.52228524e-02  1.24449073e-03 -1.25118867e-02 -5.98820858e-03
  3.79727408e-02  2.16739215e-02  8.68361326e-07 -7.54774082e-03
 -2.15119198e-02  2.46137241e-03  6.93279644e-03  1.82743045e-03
  5.53296506e-02 -5.39463386e-02  7.73292826e-03  4.64308709e-02
 -4.12593633e-02  1.71819702e-04  3.97657119e-02  2.63661444e-02
  9.22471955e-02  6.20605089e-02 -1.62736718e-02 -3.554757

In [193]:
%%time

for item in tqdm(pages_and_chunks_over_min_token_len):
    item["embedding"] = embedding_model.encode(item["sentence_chunk"])

print(pages_and_chunks_over_min_token_len[0]["embedding"])

  0%|          | 0/1680 [00:00<?, ?it/s]

[ 6.74242303e-02  9.02282074e-02 -5.09549771e-03 -3.17545831e-02
  7.39082322e-02  3.51976305e-02 -1.97986625e-02  4.67691906e-02
  5.35726734e-02  5.01229428e-03  3.33928801e-02 -1.62216206e-03
  1.76080745e-02  3.62653248e-02 -3.16640479e-04 -1.07117631e-02
  1.54257566e-02  2.62176581e-02  2.77661136e-03  3.64942625e-02
 -4.44109328e-02  1.89362243e-02  4.90117818e-02  1.64020434e-02
 -4.85782959e-02  3.18294251e-03  2.72992942e-02 -2.04754574e-03
 -1.22828772e-02 -7.28048980e-02  1.20446226e-02  1.07300421e-02
  2.10002228e-03 -8.17773417e-02  2.67830205e-06 -1.81428511e-02
 -1.20803164e-02  2.47174725e-02 -6.27467260e-02  7.35438094e-02
  2.21624803e-02 -3.28767672e-02 -1.80095695e-02  2.22952347e-02
  5.61365038e-02  1.79514487e-03  5.25931641e-02 -3.31744994e-03
 -8.33882112e-03 -1.06284758e-02  2.31918227e-03 -2.23934669e-02
 -1.53011763e-02 -9.93053615e-03  4.65322584e-02  3.57468724e-02
 -2.54760236e-02  2.63694450e-02  3.74913891e-03 -3.82680260e-02
  2.58325636e-02  4.12872

#### Batches - Computing on multiple samples at once

We dont embed each sentence, we take groups of sentences(batches) and embedd them together.

GPU has memory limitation, thats why Batch processing plays an important role



In [168]:
#turn text chunks into a single list
text_chunks = [item["sentence_chunk"] for item in pages_and_chunks_over_min_token_len]

In [ ]:
%%time

# Embed all texts in batches
text_chunk_embeddings = embedding_model.encode(
    text_chunks,
    batch_size = 32, #can use different batch sizes
    convert_to_tensor = True #optional to return embeddings as tensors
)

CPU times: user 11.8 s, sys: 1.64 s, total: 13.4 s
Wall time: 29.9 s


In [169]:
#Save embeedings to file
text_chunks_and_embeddings_df = pd.DataFrame(pages_and_chunks_over_min_token_len)

embeddings_df_save_path = "../data/text_chunks_embeddings_df.csv"
text_chunks_and_embeddings_df.to_csv(embeddings_df_save_path, index=False)

In [194]:
import random
import pandas as pd
import torch
import numpy as np

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")
#import text adn embeddings df
text_chunks_and_embeddings_df = pd.read_csv(embeddings_df_save_path)
# #convert the embeddings back to np.array (it got converted to string when it got saved to csv)
text_chunks_and_embeddings_df["embedding"] = text_chunks_and_embeddings_df["embedding"].apply(lambda x: np.fromstring(x.strip("[]"), sep=' ', dtype=np.float32 ))

# #Convert text and embeddings df to a list of dicts
pages_and_chunks = text_chunks_and_embeddings_df.to_dict(orient="records")

# #Convert embeedings to tensors and send them to device(note: Numpy arrays are float64, torch tensors are float32)
embeddings = torch.tensor(np.array(text_chunks_and_embeddings_df["embedding"].tolist()), dtype = torch.float32).to(device)
embeddings.shape

Using device: mps


torch.Size([1680, 768])

In [196]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2", device="mps")

### R - Retrieval Part

In [ ]:
from sentence_transformers import util
from time import perf_counter as timer

#Create a query
query = "macronutrients functions"
print(f"Our query: {query}")

#Encode the query
query_embedding = embedding_model.encode(query, convert_to_tensor=True)

start_time = timer()
#.dot_score returns a tensor of shape (1,n) - n dimensions(matrix)  -> we need to take the first element
dot_scores = util.dot_score(a=query_embedding, b=embeddings)[0]
print(f"[INFO] Dot product shape: {dot_scores}")

end_time = timer()

print(f'Time to get scores on {len(embeddings)} embeddings: {end_time - start_time} seconds')

#Get top k results
top_results_dot_product = torch.topk(dot_scores, k = 5)
top_results_dot_product

Our query: macronutrients functions
[INFO] Dot product shape: tensor([0.4343, 0.4406, 0.3667,  ..., 0.3941, 0.3321, 0.3707], device='mps:0')
Time to get scores on 1680 embeddings: 0.04538733302615583 seconds


torch.return_types.topk(
values=tensor([0.6926, 0.6738, 0.6646, 0.6536, 0.6473], device='mps:0'),
indices=tensor([42, 47, 41, 51, 46], device='mps:0'))

In [222]:
rez1 = top_results_dot_product[0]
print(f'Type of rez1: {type(rez1)}')
print(rez1)
rez2 = top_results_dot_product[1]
print(f'Type of rez2: {type(rez2)}')
values,indices = top_results_dot_product

zipped_results = zip(values.tolist(), indices.tolist())
print(type(zipped_results))
print(zipped_results)
zipped_results = list(zipped_results)
print(type(zipped_results))
print(zipped_results)


Type of rez1: <class 'torch.Tensor'>
tensor([0.6926, 0.6738, 0.6646, 0.6536, 0.6473], device='mps:0')
Type of rez2: <class 'torch.Tensor'>
<class 'zip'>
<class 'list'>
[(0.6925809383392334, 42), (0.6738271713256836, 47), (0.6646262407302856, 41), (0.6536345481872559, 51), (0.6472819447517395, 46)]


In [199]:
#helper function to print wrapped text
import textwrap

def print_wrapped(text, wrap_length = 80):
    wrapped_text = textwrap.fill(text, wrap_length)
    print(wrapped_text)

Now we can loop through the top_results_dot_products tuple and match up the scores and indicies and then use those indicies to index on our pages_and_chunks variable to get the relevant text chunk.

In [224]:
print(f"Query: '{query}'\n")
print(f"Results:")

for score, idx in zip(top_results_dot_product[0], top_results_dot_product[1]):
    # Print relevant sentence chunk (since the scores are in descending order, the most relevant chunk will be first)
    print(f"Score: {score:.4f}")
    print("Text:")
    print_wrapped(pages_and_chunks[idx]["sentence_chunk"])
    #Print the page number too se we can reference the textbook further (and check the results)
    print(f"Page number: {pages_and_chunks[idx]['page_number']}")
    print("\n")
    

Query: 'macronutrients functions'

Results:
Score: 0.6926
Text:
Macronutrients Nutrients that are needed in large amounts are called
macronutrients. There are three classes of macronutrients: carbohydrates,
lipids, and proteins. These can be metabolically processed into cellular energy.
The energy from macronutrients comes from their chemical bonds. This chemical
energy is converted into cellular energy that is then utilized to perform work,
allowing our bodies to conduct their basic functions. A unit of measurement of
food energy is the calorie. On nutrition food labels the amount given for
“calories” is actually equivalent to each calorie multiplied by one thousand. A
kilocalorie (one thousand calories, denoted with a small “c”) is synonymous with
the “Calorie” (with a capital “C”) on nutrition food labels. Water is also a
macronutrient in the sense that you require a large amount of it, but unlike the
other macronutrients, it does not yield calories. Carbohydrates Carbohydrates
are 

In [230]:
def dot_product(vector1, vector2):
    return torch.dot(vector1, vector2)

def cosine_similarity(vector1, vector2):
    dot_product = torch.dot(vector1, vector2)

    #Get normalization for each vector(removes the magnitude, keeps direction)
    norm_vector1 = torch.sqrt(torch.sum(vector1 ** 2))
    norm_vector2 = torch.sqrt(torch.sum(vector2 ** 2))

    #Compute cosine similarity
    cosine_similarity = dot_product / (norm_vector1 * norm_vector2)
    return cosine_similarity
    

#Example tensors
vector1 = torch.tensor([1,2,3], dtype=torch.float32)
vector2 = torch.tensor([1,2,3], dtype=torch.float32)
vector3 = torch.tensor([4,5,6], dtype=torch.float32)
vector4 = torch.tensor([-1,-2,-3], dtype=torch.float32)

#Calculate dot product
print("Dot product between vector1 and vector2: ", dot_product(vector1, vector2))
print("Dot product between vector1 and vector3: ", dot_product(vector1, vector3))
print("Dot product between vector1 and vector4: ", dot_product(vector1, vector4))

#Calculate cosine similarity
print("Cosine similarity between vector1 and vector2: ", cosine_similarity(vector1, vector2))
print("Cosine similarity between vector1 and vector3: ", cosine_similarity(vector1, vector3))
print("Cosine similarity between vector1 and vector4: ", cosine_similarity(vector1, vector4))

Dot product between vector1 and vector2:  tensor(14.)
Dot product between vector1 and vector3:  tensor(32.)
Dot product between vector1 and vector4:  tensor(-14.)
Cosine similarity between vector1 and vector2:  tensor(1.0000)
Cosine similarity between vector1 and vector3:  tensor(0.9746)
Cosine similarity between vector1 and vector4:  tensor(-1.0000)


##### Functionizing our Semantic search Pipeline


In [237]:
def retrieve_relevant_sources(query: str, embeddings: torch.Tensor, model: SentenceTransformer = embedding_model, top_k: int = 5):
    """
    Retrieve the top k most relevant sources based on the query and embeddings.
    """

    # Embed the query
    query = embedding_model.encode(query, convert_to_tensor=True)

    #Get dot product
    start_time = timer()
    dot_scores = util.dot_score(a=query, b=embeddings)[0]
    end_time = timer()

    print(f"[INFO] Time to get scores on {len(embeddings)} embeddings: {end_time - start_time} seconds")
    scores, indices = torch.topk(dot_scores, k=top_k)

    return scores, indices

    #Get top k results
    
def print_top_result_and_score(query: str, embeddings: torch.Tensor, pages_and_chunks: list[dict],top_k: int = 5):

    scores, indices = retrieve_relevant_sources(query, embeddings, top_k=top_k)

    print(f"Query: '{query}'\n")
    print(f"Results:")

    for score, idx in zip(scores, indices):
        print(f"Score: {score:.4f}")
        print("Text:")
        print_wrapped(pages_and_chunks[idx]["sentence_chunk"])
        print(f"Page number: {pages_and_chunks[idx]['page_number']}")
        print("\n")


In [239]:
query = "symptoms of pellagra"

print_top_result_and_score(query, embeddings, pages_and_chunks)

[INFO] Time to get scores on 1680 embeddings: 0.011474959086626768 seconds
Query: 'symptoms of pellagra'

Results:
Score: 0.5000
Text:
Niacin deficiency is commonly known as pellagra and the symptoms include
fatigue, decreased appetite, and indigestion.  These symptoms are then commonly
followed by the four D’s: diarrhea, dermatitis, dementia, and sometimes death.
Figure 9.12  Conversion of Tryptophan to Niacin Water-Soluble Vitamins | 565
Page number: 565


Score: 0.3741
Text:
car. Does it drive faster with a half-tank of gas or a full one?It does not
matter; the car drives just as fast as long as it has gas. Similarly, depletion
of B vitamins will cause problems in energy metabolism, but having more than is
required to run metabolism does not speed it up. Buyers of B-vitamin supplements
beware; B vitamins are not stored in the body and all excess will be flushed
down the toilet along with the extra money spent. B vitamins are naturally
present in numerous foods, and many other foods 

### G - Generation Part

In [244]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import is_flash_attn_2_available

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
#1. Pick a model
model_id = "google/gemma-2b-it"
print(f"[INFO] Loading model: {model_id}...")

#2. Instantiate tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

#3. Instantiate the model
llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype = torch.float16,
    low_cpu_mem_usage = False, #use full memory
).to(device)

[INFO] Loading model: google/gemma-2b-it...


tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/2.11G [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

Instantiating GemmaForCausalLM model under default dtype torch.float16.
Generate config GenerationConfig {
  "bos_token_id": 2,
  "eos_token_id": 1,
  "pad_token_id": 0
}



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

loading configuration file generation_config.json from cache at /Users/tybyboiciuc/.cache/huggingface/hub/models--google--gemma-2b-it/snapshots/96988410cbdaeb8d5093d1ebdc5a8fb563e02bad/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 2,
  "eos_token_id": 1,
  "pad_token_id": 0
}

Could not locate the custom_generate/generate.py inside google/gemma-2b-it.


In [ ]:
def get_model_num_parameters(model: torch.nn.Module) -> int:

    return sum([p.numel() for p in model.parameters()])

get_model_num_parameters(llm_model)

GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): Embedding(256000, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x GemmaDecoderLayer(
        (self_attn): GemmaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): GemmaMLP(
          (gate_proj): Linear(in_features=2048, out_features=16384, bias=False)
          (up_proj): Linear(in_features=2048, out_features=16384, bias=False)
          (down_proj): Linear(in_features=16384, out_features=2048, bias=False)
          (act_fn): GELUActivation()
        )
        (input_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): GemmaRMSNorm((2048,), 

In [249]:
def get_model_mem_size(model: torch.nn.Module):
    """
    Get how much memory a pytorch model takes up
    """

    mem_params = sum([param.nelement() * param.element_size() for param in model.parameters()])
    mem_buffers = sum([buf.nelement() * buf.element_size() for buf in model.buffers()])

    #Calculate various model sizes
    model_mem_bytes = mem_params + mem_buffers #in bytes
    model_mem_mb = model_mem_bytes / (1024 ** 2) #in MB
    model_mem_gb = model_mem_bytes / (1024 ** 3) #in GB

    return {
        "model_mem_bytes": model_mem_bytes,
        "model_mem_mb": round(model_mem_mb, 2),
        "model_mem_gb": round(model_mem_gb, 2)
    }

get_model_mem_size(llm_model)

{'model_mem_bytes': 5012345344, 'model_mem_mb': 4780.15, 'model_mem_gb': 4.67}

##### Generating a text with our LLM

In [263]:
input_text = "What are the macronutrients, and what roles do they play in the human body?"
print(f'Input text: {input_text}')

#Create prompt template for instruction-tuned model
dialogue_template = [
    {"role": "user",
     "content":input_text}
]

# Apply the chat template
prompt = tokenizer.apply_chat_template(conversation=dialogue_template,
                                       tokenize=False, #keep as raw text(not tokenized)
                                       add_generation_prompt=True,
                                       add_special_tokens=True)

print(f'\nPrompt: {prompt}')

Input text: What are the macronutrients, and what roles do they play in the human body?

Prompt: <bos><start_of_turn>user
What are the macronutrients, and what roles do they play in the human body?<end_of_turn>
<start_of_turn>model



In [284]:
%%time

# Tokenize the input text (turn in into numbers) and send it to GPU
input_ids = tokenizer(prompt, return_tensors="pt").to(device)
print(f'Model input (tokenized):\n {input_ids}\n')

# Generate outputs passed on the tokenized input
outputs = llm_model.generate(**input_ids, max_new_tokens = 256) #maxim number of tokens to create

print(f"Model output (tokens): {outputs[0]}")

Model input (tokenized):
 {'input_ids': tensor([[     2,      2,    106,   1645,    108,   1841,    708,    573, 186809,
         184592, 235269,    578,   1212,  16065,    749,    984,   1554,    575,
            573,   3515,   2971, 235336,    107,    108,    106,   2516,    108]],
       device='mps:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1]], device='mps:0')}

Model output (tokens): tensor([     2,      2,    106,   1645,    108,   1841,    708,    573, 186809,
        184592, 235269,    578,   1212,  16065,    749,    984,   1554,    575,
           573,   3515,   2971, 235336,    107,    108,    106,   2516,    108,
         21404, 235269,   1517, 235303, 235256,    476,  25497,    576,    573,
        186809, 184592,    578,   1024,  16065,    575,    573,   3515,   2971,
        235292,    109,    688,  12298,   1695, 184592,  66058,    109, 235287,
          5231, 156615,  56227,  66058,    108,    

In [ ]:
#Decode the output tokens to text
outputs_decoded = tokenizer.decode(outputs[0])
print(f"Model output (decoded): {outputs_decoded}")

Model output (decoded): <bos><bos><start_of_turn>user
What are the macronutrients, and what roles do they play in the human body?<end_of_turn>
<start_of_turn>model
Sure, here's a breakdown of the macronutrients and their roles in the human body:

**Macronutrients:**

* **Carbohydrates:**
    * Provide energy for the body's cells and tissues.
    * Carbohydrates are the primary source of energy for most cells.
    * Complex carbohydrates are those that take longer to digest, such as whole grains, fruits, and vegetables.
    * Simple carbohydrates are those that are quickly digested, such as sugar, starch, and lactose.

* **Proteins:**
    * Build and repair tissues, enzymes, and hormones.
    * Proteins are essential for immune function, hormone production, and tissue repair.
    * There are different types of proteins, each with specific functions.

* **Fats:**
    * Provide energy, insulation, and help absorb vitamins.
    * Healthy fats include olive oil, avocado, nuts, and seeds.
  

In [ ]:
print(f"Input text: {input_text}")
print(f"Output text: {outputs_decoded}")